# RecoverAI — Model Training & EDA

This notebook documents the training data distribution, feature analysis, and model selection process for the V3 recoverability model.

The trained artifact is saved to `models/recovery_probability_model_v3.joblib`.
The full evaluation report is at `models/recovery_model_v3_report.json`.

> **Note:** All data is synthetic. Outcomes are simulated for benchmark reproducibility. No real customer money is involved.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import numpy as np

## 1. Training Data

In [ ]:
df = pd.read_csv("../data/processed/training_data.csv")
print(f"Rows: {len(df):,}")
df.head()

In [ ]:
print("Recovery rate (training):", df["recovered"].mean().round(4))
print()
print("Failure category distribution:")
print(df["failure_category"].value_counts())

In [ ]:
print("Recovery rate by failure category:")
df.groupby("failure_category")["recovered"].mean().sort_values(ascending=False).round(3)

In [ ]:
print("Recovery rate by action:")
df.groupby("action")["recovered"].mean().sort_values(ascending=False).round(3)

In [ ]:
print("Recovery rate by customer segment:")
df.groupby("customer_segment")["recovered"].mean().sort_values(ascending=False).round(3)

## 2. Unseen Evaluation Population

In [ ]:
agent = pd.read_csv("../data/unseen/processed/agent_results.csv")
baseline = pd.read_csv("../data/unseen/processed/baseline_results.csv")

print(f"Unseen cases: {len(agent):,}")
print(f"RecoverAI recovery rate:  {agent['final_status'].eq('recovered').mean():.2%}")
print(f"Baseline recovery rate:   {baseline['recovered'].mean():.2%}")
print(f"RecoverAI recovered:      ₹{agent['total_recovered'].sum():,.2f}")
print(f"Baseline recovered:       ₹{baseline['recovered_amount'].sum():,.2f}")

## 3. V3 Model Report

In [ ]:
report = json.loads(Path("../models/recovery_model_v3_report.json").read_text())
print(f"Model version:   {report['model_version']}")
print(f"Training rows:   {report['rows']:,}")
print(f"Positive rate:   {report['positive_rate']:.2%}")
print()
print("Selected model metrics:")
for k, v in report["selected_model"].items():
    if k != "model":
        print(f"  {k}: {v:.4f}")

## 4. Policy Safety Verification

The model never overrides policy. These checks confirm the guardrails hold on the unseen population.

In [ ]:
decisions = pd.read_csv("../data/unseen/processed/decisions.csv")

risk_retried = decisions[
    (decisions["failure_category"] == "risk_decline") &
    (decisions["final_action"] == "retry")
]
print(f"risk_decline cases retried (must be 0): {len(risk_retried)}")

print(f"Final action distribution:")
print(decisions["final_action"].value_counts())